# Vision · 범위 제한 기반 빈칸·부분 구현 문제

원본에 마스킹이 없어 **완성된 실습 코드를 바탕으로 새로 선정한 연습 문제**입니다. 실제 시험 문항이나 공식 배점을 의미하지 않습니다.

- ResNet: **0~5절**, 전체 셀15까지. 이번 요청의 “Chapter 6 이전”을 적용하여 6절 자체도 제외합니다.
- ViT: **6절 시작 전**, 전체 셀23까지. 원본 절3/4 순서가 뒤섞여 있어 셀 번호도 함께 기록합니다.
- DETR: 기존 확정 범위인 **13절 포함**, 전체 셀30까지.
- U-Net: 지정 노트북 전체. 설치·다운로드·긴 학습·그래프 꾸미기는 제외합니다.

근거: [시험 출제 포인트 PDF](../../AI인증시험_출제포인트.pdf) 1쪽 Vision의 레이어 이해, 예측/정답을 학습으로 연결, 데이터 구성, 이미지 변환 라이브러리, 평가 Metric. 문항별 셀 번호는 **Markdown 포함,1부터**입니다.

**사용법:** 준비 셀을 먼저 실행하고 각 문제에서 `MISSING` 또는 `raise NotImplementedError`를 올바른 코드로 교체 → 바로 아래 테스트 실행 → 실패하면 shape/축/순서 확인 → 마지막의 접힌 해설 확인. `# ????`는 원본에서 가져온 빈칸이 아니라 새 마스킹 표시입니다. 각 문제는 독립되어 이전 문제의 답이 필요 없습니다. 답을 수정하면 함수 셀을 먼저 다시 실행하세요. 예상 5~10분/문제(연습용 가안). 작성한 이 노트북을 제출하면 채점할 수 있습니다.

CPU·PyTorch·torchvision·einops·NumPy·Pillow만 사용합니다. 모델/데이터 다운로드 없음. 테스트는 shape뿐 아니라 수치, 학습 갱신, 잘못된 축, 다른 인자도 검사합니다. 통과는 작은 입력 검증이며 실전 정확도나 독립 숙달을 보장하지 않습니다.

**원본과 차이:** ViT FFN은 실제 코드의 ReLU 사용(주석의 GELU와 다름). U-Net DoubleConv는 실제 코드처럼 BN 없음. DETR attention reshape는 query/key 축이 뒤바뀐 설명·슬라이스를 교정하고 문제에서 명시합니다. ResNet AMP는 CPU에서 비활성 scaler로 순서를 확인합니다. 설치된 PyTorch 2.5.1/torchvision 0.20.1/einops 0.8.2에서 검증합니다.

이미지 텐서는 PyTorch의 실제 축 순서 **B,C,H,W**로 표기합니다. W,H를 구분하는 비정사각형 입력도 검사합니다. 이해가 안 되는 문제 번호·줄을 알려주면 작은 행렬 예시로 다시 설명할 수 있습니다.


In [ ]:
import copy
import math
import numpy as np
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

torch.set_num_threads(1)
MISSING = object()  # ???? 대신 실행 가능한 표식. 이 값을 올바른 코드로 바꾸세요.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)

def close(actual, expected):
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)

class ModeProbe(nn.Module):
    # train/eval과 gradient 모드를 기록하는 작은 제공 분류기입니다.
    def __init__(self, width=4, classes=3):
        super().__init__()
        self.fc = nn.Linear(width, classes)
        self.seen = []
    def forward(self, x):
        self.seen.append((self.training, torch.is_grad_enabled()))
        return self.fc(x)


## V01. 이미지 패치화와 선형 임베딩

**PDF 점검:** 레이어 구성·데이터 구성 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 11

이미지의 패치를 (p1,p2,c) 순서로 펼친 후 Linear(patch_dim,dim)를 적용하는 Sequential을 만드세요. 입력 (B,C,H,W), 패치(ph,pw), 출력 (B,N,dim). H,W는 패치 크기로 나누어집니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v01(channels, ph, pw, dim):
    patch_dim = MISSING
    return nn.Sequential(
        Rearrange(MISSING,p1=ph,p2=pw),
        MISSING,
    )

In [ ]:
def check_v01(fn):
    for c,ph,pw,h,w,d in [(3,2,2,4,4,2),(2,2,3,4,6,5)]:
        layer=fn(c,ph,pw,d)
        x=torch.arange(2*c*h*w).float().reshape(2,c,h,w)
        patches=[]
        for y in range(0,h,ph):
            for z in range(0,w,pw):patches.append(x[:,:,y:y+ph,z:z+pw].permute(0,2,3,1).reshape(2,-1))
        expected=torch.stack(patches,dim=1)
        close(layer[0](x),expected)
        assert layer[1].in_features==c*ph*pw and layer[1].out_features==d
        close(layer(x),F.linear(expected,layer[1].weight,layer[1].bias))

try:
    check_v01(solve_v01)
    print("PASS · V01")
except Exception as error:
    print("FAIL · V01 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: (B,3,4,4),패치2×2 → 패치4개,패치당12개 숫자 → (B,4,12)→Linear(12,2)→(B,4,2). Rearrange는 값의 순서·shape만 바꾸고 Linear가 학습 가능한 투영을 합니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v01(channels, ph, pw, dim):
    patch_dim = channels * ph * pw
    return nn.Sequential(
        Rearrange("b c (h p1) (w p2) -> b (h w) (p1 p2 c)",p1=ph,p2=pw),
        nn.Linear(patch_dim,dim),
    )
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V02. CLS 토큰과 위치 임베딩

**PDF 점검:** 레이어 구성·동작 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 11

x=(B,N,D), cls_token=(1,1,D), pos_embedding=(1,M,D), M>=N+1입니다. CLS를 배치만큼 복제하여 토큰 축 맨 앞에 붙이고 필요한 위치 임베딩만 더하세요.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v02(x, cls_token, pos_embedding):
    b,n,_=x.shape
    cls_tokens=MISSING
    x=MISSING
    return MISSING

In [ ]:
def check_v02(fn):
    for b,n,d in [(1,4,2),(3,6,4)]:
        x=torch.arange(b*n*d).float().reshape(b,n,d)
        cls=torch.full((1,1,d),-2.,requires_grad=True)
        pos=torch.arange((n+3)*d).float().reshape(1,n+3,d).requires_grad_()
        got=fn(x,cls,pos)
        expected=torch.stack([torch.vstack([cls[0],x[i]])+pos[0,:n+1] for i in range(b)])
        close(got,expected);assert got.shape==(b,n+1,d)
        got.sum().backward();close(cls.grad,torch.full_like(cls,float(b)))

try:
    check_v02(solve_v02)
    print("PASS · V02")
except Exception as error:
    print("FAIL · V02 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: 패치4개×차원2 → CLS 추가 후5개×차원2. 위치0은 CLS, 위치1~4는 패치입니다. 위치 임베딩은 새 토큰으로 cat하는 것이 아니라 각 위치 벡터에 더합니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v02(x, cls_token, pos_embedding):
    b,n,_=x.shape
    cls_tokens=repeat(cls_token,"1 1 d -> b 1 d",b=b)
    x=torch.cat((cls_tokens,x),dim=1)
    return x + pos_embedding[:,:(n+1)]
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V03. PreNorm: 먼저 LayerNorm

**PDF 점검:** 레이어 구성·동작 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 9

PreNorm의 forward 일부입니다. 제공된 norm은 LayerNorm(D), fn은 D→D 모듈입니다. 마지막 임베딩 축을 정규화한 뒤 fn에 전달하세요.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v03(x, norm, fn):
    return MISSING

In [ ]:
def check_v03(fn):
    for d in (2,5):
        torch.manual_seed(d);x=torch.randn(2,3,d);norm=nn.LayerNorm(d);layer=nn.Linear(d,d)
        got=fn(x,norm,layer)
        z=(x-x.mean(-1,keepdim=True))/torch.sqrt(x.var(-1,keepdim=True,unbiased=False)+norm.eps)
        close(got,F.linear(z*norm.weight+norm.bias,layer.weight,layer.bias))

try:
    check_v03(solve_v03)
    print("PASS · V03")
except Exception as error:
    print("FAIL · V03 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: 각 토큰 [1,3]을 먼저 평균0 근처로 정규화하고 Attention/FFN에 넣습니다. shape (B,N,D)는 유지됩니다. fn(norm(x))와 norm(fn(x))는 순서가 달라 결과도 다릅니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v03(x, norm, fn):
    return fn(norm(x))
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V04. FFN의 확장·활성화·축소

**PDF 점검:** 레이어 구성·동작 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 9

Linear(dim,hidden_dim)→ReLU→Dropout→Linear(hidden_dim,dim)→Dropout의 FFN을 만드세요. 주의: 지정 원본의 주석은 GELU지만 실제 코드는 ReLU이므로 이 문제는 ReLU를 요구합니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v04(dim, hidden_dim, dropout):
    return nn.Sequential(
        MISSING,
        MISSING,
        nn.Dropout(dropout),
        MISSING,
        nn.Dropout(dropout),
    )

In [ ]:
def check_v04(fn):
    for d,hidden,p in [(2,5,.2),(4,7,.4)]:
        m=fn(d,hidden,p).eval()
        assert [type(a) for a in m]==[nn.Linear,nn.ReLU,nn.Dropout,nn.Linear,nn.Dropout]
        assert m[2].p==m[4].p==p
        x=torch.randn(2,3,d)
        assert m[0](x).shape==(2,3,hidden)
        expected=F.linear(F.relu(F.linear(x,m[0].weight,m[0].bias)),m[3].weight,m[3].bias)
        close(m(x),expected);assert m(x).shape==x.shape

try:
    check_v04(solve_v04)
    print("PASS · V04")
except Exception as error:
    print("FAIL · V04 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: (2,5,4)→Linear(4,8)→(2,5,8)→ReLU→Linear(8,4)→(2,5,4). 토큰 개수5는 유지됩니다. 여기에는 LayerNorm을 중복 삽입하지 않고 바깥 PreNorm이 담당합니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v04(dim, hidden_dim, dropout):
    return nn.Sequential(
        nn.Linear(dim,hidden_dim),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden_dim,dim),
        nn.Dropout(dropout),
    )
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V05. QKV 분리와 head 축 만들기

**PDF 점검:** 레이어 구성·동작 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 9

to_qkv는 D→3*heads*dim_head인 Linear입니다. x=(B,N,D)를 투영하고 마지막 축을 3등분한 뒤 각각 (B,heads,N,dim_head)로 바꾸세요. Q,K,V 순서로 반환합니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v05(x, to_qkv, heads):
    qkv=MISSING
    q,k,v=map(lambda t: MISSING,qkv)
    return q,k,v

In [ ]:
def check_v05(fn):
    for b,n,d,h,dh in [(2,5,4,2,3),(1,4,3,3,2)]:
        x=torch.randn(b,n,d);layer=nn.Linear(d,3*h*dh,bias=False)
        got=fn(x,layer,h); raw=F.linear(x,layer.weight)
        for i,t in enumerate(got):
            expected=raw[:,:,i*h*dh:(i+1)*h*dh].reshape(b,n,h,dh).permute(0,2,1,3)
            close(t,expected);assert t.shape==(b,h,n,dh)

try:
    check_v05(solve_v05)
    print("PASS · V05")
except Exception as error:
    print("FAIL · V05 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: B2,N5,D4,heads2,dim_head3이면 Linear 출력(2,5,18)→Q,K,V 각(2,5,6)→각(2,2,5,3). 입력D와 heads×dim_head가 꼭 같아야 하는 구현이 아닙니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v05(x, to_qkv, heads):
    qkv=to_qkv(x).chunk(3,dim=-1)
    q,k,v=map(lambda t: rearrange(t,"b n (h d) -> b h n d",h=heads),qkv)
    return q,k,v
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V06. Scaled attention과 V 가중합

**PDF 점검:** 레이어 구성·동작 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 9

q,k,v=(B,H,N,d)입니다. QKᵀ에 d**-0.5를 곱하고 key축 softmax를 적용한 뒤 V를 가중합하세요. 반환 attn=(B,H,N,N),out=(B,H,N,d). ViT에는 causal mask를 넣지 않습니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v06(q,k,v):
    scale=MISSING
    dots=MISSING
    attn=MISSING
    out=MISSING
    return attn,out

In [ ]:
def check_v06(fn):
    for b,h,n,d in [(1,2,3,2),(2,3,4,3)]:
        torch.manual_seed(n);q=torch.randn(b,h,n,d);k=torch.randn_like(q);v=torch.randn_like(q)
        attn,out=fn(q,k,v)
        scores=(q@k.transpose(-2,-1))/math.sqrt(d)
        expected=(scores-scores.max(-1,keepdim=True).values).exp()
        expected=expected/expected.sum(-1,keepdim=True)
        close(attn,expected);close(out,expected@v)
        close(attn.sum(-1),torch.ones(b,h,n));assert (attn>0).all()

try:
    check_v06(solve_v06)
    print("PASS · V06")
except Exception as error:
    print("FAIL · V06 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: 한 query의 weight=[.2,.8], V=[[1,0],[0,3]]이면 새 벡터=[.2,2.4]. i는 query, j는 key, d는 내적하는 특징축입니다. softmax 전 값은 score이고 후가 가중치입니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v06(q,k,v):
    scale=q.shape[-1] ** -0.5
    dots=torch.einsum("b h i d,b h j d->b h i j",q,k) * scale
    attn=dots.softmax(dim=-1)
    out=torch.einsum("b h i j,b h j d->b h i d",attn,v)
    return attn,out
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V07. head 합치기와 출력 투영

**PDF 점검:** 레이어 구성·동작 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 9

out=(B,H,N,d)를 (B,N,H*d)로 재배열하고 제공된 to_out으로 투영하세요. heads를 섞지 않게 n과 h 축 순서를 주의하세요.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v07(out,to_out):
    out=MISSING
    return MISSING

In [ ]:
def check_v07(fn):
    for b,h,n,d,od in [(2,2,3,2,3),(1,3,4,2,5)]:
        x=torch.arange(b*h*n*d).float().reshape(b,h,n,d)
        layer=nn.Linear(h*d,od)
        expected=F.linear(x.permute(0,2,1,3).reshape(b,n,h*d),layer.weight,layer.bias)
        close(fn(x,layer),expected)

try:
    check_v07(solve_v07)
    print("PASS · V07")
except Exception as error:
    print("FAIL · V07 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: (2,2,5,3)→(2,5,6)→Linear(6,4)→(2,5,4). 단순 reshape만 하면 토큰/헤드의 숫자 순서가 잘못 섞일 수 있습니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v07(out,to_out):
    out=rearrange(out,"b h n d -> b n (h d)")
    return to_out(out)
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V08. 두 residual 연결의 순서

**PDF 점검:** 레이어 구성·동작 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 9

attn과 ff에는 PreNorm이 이미 포함돼 있습니다. attention 결과에 입력을 더하고, 갱신된 x로 FFN을 계산해 다시 잔차를 더하세요. shape는 (B,N,D)로 유지됩니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v08(x,attn,ff):
    x=MISSING
    x=MISSING
    return x

In [ ]:
def check_v08(fn):
    for d in (2,4):
        x=torch.linspace(-2,2,2*3*d).reshape(2,3,d)
        a=lambda t:t*2+1
        f=lambda t:t.square()
        mid=3*x+1
        close(fn(x,a,f),mid.square()+mid)

try:
    check_v08(solve_v08)
    print("PASS · V08")
except Exception as error:
    print("FAIL · V08 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: x=1,attn(x)=2x이면 첫 결과3. ff(t)=t²이면 두 번째 결과9+3=12. 두 번째 skip의 x는 최초1이 아니라 갱신된3입니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v08(x,attn,ff):
    x=attn(x)+x
    x=ff(x)+x
    return x
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V09. CLS/mean pooling과 클래스 logits

**PDF 점검:** 레이어 구성·동작 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 11

x=(B,N+1,D)에서 pool="cls"면0번 토큰, "mean"이면 토큰축 평균을 선택합니다. 제공 mlp_head(LayerNorm→Linear)에 넣어 (B,C) logits를 반환하세요. 원본 mean은 CLS도 포함합니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v09(x,pool,mlp_head):
    x=MISSING
    return MISSING

In [ ]:
def check_v09(fn):
    for pool in ('cls','mean'):
        x=torch.randn(2,5,4);head=nn.Sequential(nn.LayerNorm(4),nn.Linear(4,3))
        picked=x[:,0,:] if pool=='cls' else sum(x[:,i,:] for i in range(5))/5
        close(fn(x,pool,head),head(picked));assert fn(x,pool,head).shape==(2,3)

try:
    check_v09(solve_v09)
    print("PASS · V09")
except Exception as error:
    print("FAIL · V09 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: (2,5,4)에서 CLS 선택→(2,4)→head→(2,10). 마지막 softmax는 분류 loss 입력 전에 넣지 않습니다. 토큰마다 클래스10개를 출력하는 문제가 아닙니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v09(x,pool,mlp_head):
    x=x[:,0] if pool=="cls" else x.mean(dim=1)
    return mlp_head(x)
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V10. log_softmax와 NLLLoss 학습

**PDF 점검:** 예측·정답을 학습으로 연결 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 19

원본 ViT의 한 배치 학습 코드를 완성하세요. logits에 log_softmax(dim=1)를 적용한 뒤 nll_loss를 사용합니다. 반환값은 float loss. 모델·optimizer·데이터는 제공됩니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v10(model,optimizer,data,target):
    model.train()
    optimizer.zero_grad()
    output=MISSING
    loss=MISSING
    raise NotImplementedError("???? 이 줄을 완성하세요")
    raise NotImplementedError("???? 이 줄을 완성하세요")
    return loss.item()

In [ ]:
def check_v10(fn):
    for seed in (3,6):
        torch.manual_seed(seed);a=ModeProbe();b=copy.deepcopy(a);a.eval()
        oa=torch.optim.SGD(a.parameters(),lr=.02);ob=torch.optim.SGD(b.parameters(),lr=.02)
        for size in (2,3):
            x=torch.randn(size,4);y=torch.arange(size)%3
            ob.zero_grad();loss=F.cross_entropy(b(x),y);loss.backward();ob.step()
            assert abs(fn(a,oa,x,y)-loss.item())<1e-6
            for p,q in zip(a.parameters(),b.parameters()):close(p,q)
        assert all(t==(True,True) for t in a.seen)

try:
    check_v10(solve_v10)
    print("PASS · V10")
except Exception as error:
    print("FAIL · V10 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: 정답 클래스 확률.8이면 NLL=-log(.8)。log_softmax+NLLLoss는 동일 logits에 CrossEntropyLoss를 적용한 것과 같습니다. softmax 확률을 nll_loss에 그대로 넣으면 다릅니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v10(model,optimizer,data,target):
    model.train()
    optimizer.zero_grad()
    output=F.log_softmax(model(data),dim=1)
    loss=F.nll_loss(output,target)
    loss.backward()
    optimizer.step()
    return loss.item()
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## V11. 체크포인트에서 학습 상태 복원

**PDF 점검:** 학습에 필요한 작업·레이어 활용 · **근거:** [02_ViT_CIFAR10.ipynb](../../vision/02_ViT_CIFAR10.ipynb) 전체 셀 15, 23

이미 메모리에 읽어 둔 ckpt를 제공합니다. model과 optimizer의 state_dict를 복원하고 epoch와 test_acc를 반환하세요. 파일 읽기·저장은 출제하지 않습니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_v11(model,optimizer,ckpt):
    raise NotImplementedError("???? 이 줄을 완성하세요")
    raise NotImplementedError("???? 이 줄을 완성하세요")
    return int(ckpt.get("epoch",0)),float(ckpt.get("test_acc",0.0))

In [ ]:
def check_v11(fn):
    for epoch in (2,7):
        a=nn.Linear(2,3);opt=torch.optim.SGD(a.parameters(),lr=.07,momentum=.8)
        F.cross_entropy(a(torch.ones(2,2)),torch.tensor([0,1])).backward();opt.step()
        ckpt=copy.deepcopy(dict(model=a.state_dict(),optimizer=opt.state_dict(),epoch=epoch,test_acc=.6))
        b=nn.Linear(2,3);ob=torch.optim.SGD(b.parameters(),lr=.01,momentum=.0)
        assert fn(b,ob,ckpt)==(epoch,.6)
        for p,q in zip(a.parameters(),b.parameters()):close(p,q)
        assert ob.param_groups[0]['lr']==.07 and ob.param_groups[0]['momentum']==.8
        for p,q in zip(a.parameters(),b.parameters()):close(opt.state[p]['momentum_buffer'],ob.state[q]['momentum_buffer'])

try:
    check_v11(solve_v11)
    print("PASS · V11")
except Exception as error:
    print("FAIL · V11 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: epoch7,lr=.07,모멘텀 버퍼를 저장했다면 모델 weight뿐 아니라 optimizer 상태도 복원해야 그 상태에서 학습을 이어갑니다. 이 문제의 입력은 다운로드 없는 dict입니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

def solve_v11(model,optimizer,ckpt):
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    return int(ckpt.get("epoch",0)),float(ckpt.get("test_acc",0.0))
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>